In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon, spearmanr
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULT_PATH = Path("/root/results/bhm_final_fix_beta_26/")
FIG_PATH    = Path("/root/figures/behavioral_final/")
FIG_PATH.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-white')
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         9,
    'axes.linewidth':    1.0,
    'axes.labelsize':    9,
    'axes.titlesize':    9,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size':  3.5,
    'ytick.major.size':  3.5,
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'legend.fontsize':   8,
    'legend.frameon':    False,
    'figure.dpi':        300,
    'pdf.fonttype':      42,    # editable text in Illustrator
    'svg.fonttype':      'none', # editable text in Inkscape
    'ps.fonttype':       42,
})

C_EQUAL   = '#1D9E75'
C_UNEQUAL = '#D85A30'
C_BOTH    = '#888780'
C_FIT     = '#2D2D2D'
C_OMEGA   = '#185FA5'
C_PROB    = '#534AB7'


# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING  (identical to previous version)
# ─────────────────────────────────────────────────────────────────────────────

def load_all_data():
    metric_files = sorted(RESULT_PATH.glob("*_game_metrics.csv"))
    param_files  = sorted(RESULT_PATH.glob("*_subject_params.csv"))
    if not metric_files:
        raise FileNotFoundError(f"No *_game_metrics.csv in {RESULT_PATH}")
    print(f"Found {len(metric_files)} metric files, {len(param_files)} param files")

    # ── subject-level params ──────────────────────────────────────────────
    subj_rows = []
    for pf in param_files:
        subj_id = pf.stem.replace('_subject_params', '')
        try:
            ps  = pd.read_csv(pf, index_col=0)
            row = {'subject': subj_id}
            for param in ['alpha_subj', 'gamma_subj', 'omega_subj', 'beta_subj']:
                if param in ps.index:
                    row[f'{param}_mean']   = float(ps.loc[param, 'mean'])
                    row[f'{param}_sd']     = float(ps.loc[param, 'sd'])
                    row[f'{param}_hdi_lo'] = float(ps.loc[param, 'hdi_3%'])
                    row[f'{param}_hdi_hi'] = float(ps.loc[param, 'hdi_97%'])
                    row[f'{param}_rhat']   = float(ps.loc[param, 'r_hat'])
            subj_rows.append(row)
        except Exception as e:
            print(f"  [WARN] {pf.name}: {e}")

    subj_df = (pd.DataFrame(subj_rows) if subj_rows
               else pd.DataFrame(columns=['subject']))

    # ── per-game trials ───────────────────────────────────────────────────
    trial_rows = []
    for mf in metric_files:
        subj_id = mf.stem.replace('_game_metrics', '')
        try:
            gm = pd.read_csv(mf)
        except Exception as e:
            print(f"  [WARN] {mf.name}: {e}")
            continue

        for _, row in gm.iterrows():
            V = np.array([row['V_Deck_1'], row['V_Deck_2'], row['V_Deck_3']], dtype=float)
            Q = np.array([row['Q_Deck_1'], row['Q_Deck_2'], row['Q_Deck_3']], dtype=float)
            I = np.array([row['I_Deck_1'], row['I_Deck_2'], row['I_Deck_3']], dtype=float)

            chosen_idx = int(row['Chosen_Deck']) - 1
            if not (0 <= chosen_idx <= 2):
                continue

            # Condition from I distribution
            I_std     = float(np.std(I))
            condition = 'equal' if I_std < 0.5 else 'unequal'

            max_V_idx   = int(np.argmax(V))
            max_Q_idx   = int(np.argmax(Q))
            chose_max_V = int(chosen_idx == max_V_idx)
            chose_max_Q = int(chosen_idx == max_Q_idx)

            # Conflict trial: argmax(V) ≠ argmax(Q)
            # Only possible when I differs across decks (unequal condition)
            is_conflict = int(max_V_idx != max_Q_idx)

            others_V = np.delete(V, chosen_idx)
            others_Q = np.delete(Q, chosen_idx)
            others_I = np.delete(I, chosen_idx)

            trial_rows.append({
                'subject':       subj_id,
                'game':          int(row['Game']),
                'condition':     condition,
                'chosen_idx':    chosen_idx,
                'chose_max_V':   chose_max_V,
                'chose_max_Q':   chose_max_Q,
                'is_conflict':   is_conflict,
                'chosen_prob':   float(row['Chosen_Prob']),
                'delta_V':       float(V[chosen_idx] - others_V.mean()),
                'delta_Q':       float(Q[chosen_idx] - others_Q.mean()),
                'delta_I_info':  float(others_I.mean() - I[chosen_idx]),
            })

    trial_df = pd.DataFrame(trial_rows)

    # Merge explore rate
    er = (trial_df.groupby('subject')['chose_max_V']
          .mean().reset_index()
          .rename(columns={'chose_max_V': 'explore_rate'}))
    subj_df = (subj_df.merge(er, on='subject', how='outer')
               if not subj_df.empty else er)

    print(f"Loaded {len(trial_df):,} trials | "
          f"{trial_df['subject'].nunique()} subjects")
    n_conflict = trial_df.query("condition=='unequal' and is_conflict==1")
    print(f"Conflict trials (unequal, argmax-V ≠ argmax-Q): {len(n_conflict):,}")
    return trial_df, subj_df


# ─────────────────────────────────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def export_stats(stats, out_path):
    out_path.mkdir(parents=True, exist_ok=True)

    # JSON（结构化）
    with open(out_path / "behavior_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    # CSV（平铺）
    rows = []
    for k, v in stats.items():
        if v is None:
            continue
        for subk, val in v.items():
            rows.append({'analysis': k, 'metric': subk, 'value': val})

    pd.DataFrame(rows).to_csv(out_path / "behavior_stats.csv", index=False)

    # TXT（论文用）
    with open(out_path / "behavior_stats.txt", "w") as f:
        for k, v in stats.items():
            f.write(f"\n[{k.upper()}]\n")
            if v is None:
                f.write("None\n")
                continue
            for subk, val in v.items():
                f.write(f"{subk}: {val}\n")
                
def sigmoid_2p(x, k, x0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

def cohen_d_paired(a, b):
    diff = np.asarray(a) - np.asarray(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-12)

def bracket(ax, x1, x2, y, p, d=None, fs=7.5):
    ylim = ax.get_ylim()
    h = (ylim[1] - ylim[0]) * 0.025
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=0.8, color='k')
    label = sig_stars(p) + (f'\nd={d:.2f}' if d is not None else '')
    ax.text((x1+x2)/2, y + h*1.1, label,
            ha='center', va='bottom', fontsize=fs, color='k')

def raincloud(ax, data, pos, color, width=0.30, jitter=0.06):
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    if len(data) < 2:
        return
    vp   = ax.violinplot([data], positions=[pos], widths=width,
                         showmeans=False, showmedians=False, showextrema=False)
    body = vp['bodies'][0]
    body.set_facecolor(color); body.set_edgecolor('none'); body.set_alpha(0.40)
    verts = body.get_paths()[0].vertices
    mid   = np.mean(verts[:, 0])
    verts[:, 0] = np.clip(verts[:, 0], mid, np.inf)

    q1, med, q3 = np.percentile(data, [25, 50, 75])
    iqr = q3 - q1
    lo  = max(data.min(), q1 - 1.5*iqr)
    hi  = min(data.max(), q3 + 1.5*iqr)
    bx  = pos - width*0.20
    ax.plot([bx]*2, [lo, hi], color=color, lw=1.0, zorder=3)
    ax.plot([bx]*2, [q1, q3], color=color, lw=3.2, zorder=4)
    ax.plot(bx, med, 'o', color='white', markeredgecolor=color,
            markersize=4.5, zorder=5)
    rng = np.random.default_rng(42)
    xs  = pos - width*0.58 + rng.uniform(-jitter, jitter, size=len(data))
    ax.scatter(xs, data, color=color, alpha=0.45, s=9, zorder=2, linewidths=0)

def despine(ax):
    sns.despine(ax=ax, top=True, right=True)
    ax.tick_params(direction='out', length=3)

def save_panel(fig, stem):
    """Save a single-panel figure as both PDF and SVG."""
    for ext in ('pdf', 'svg'):
        out = FIG_PATH / f"{stem}.{ext}"
        fig.savefig(out, bbox_inches='tight',
                    facecolor='white', transparent=False)
        print(f"  → {out}")


# ─────────────────────────────────────────────────────────────────────────────
#  PANEL A — Psychometric curve
# ─────────────────────────────────────────────────────────────────────────────

def make_panel_A(trial_df):
    fig, ax = plt.subplots(figsize=(2.5, 2.5))

    df = trial_df[['delta_V', 'chose_max_V']].dropna().copy()
    df['bin'] = pd.qcut(df['delta_V'], q=28, duplicates='drop')
    bs = df.groupby('bin', observed=True).agg(
        x=('delta_V',    'mean'),
        y=('chose_max_V','mean'),
        n=('chose_max_V','count'),
    ).reset_index()
    bs['se'] = np.sqrt(bs['y'] * (1 - bs['y']) / bs['n'].clip(lower=1))

    ax.errorbar(bs['x'], bs['y'], yerr=bs['se'],
                fmt='o', color=C_EQUAL, markersize=3.5, alpha=0.75,
                elinewidth=0.7, capsize=2.0, capthick=0.7, zorder=3,
                label='Observed ± s.e.m.')

    x_arr = df['delta_V'].values
    y_arr = df['chose_max_V'].values.astype(float)
    try:
        popt, pcov = curve_fit(sigmoid_2p, x_arr, y_arr,
                               p0=[0.25, 0.0], maxfev=10000,
                               bounds=([0.001, -50], [5.0, 50]))
        perr  = np.sqrt(np.diag(pcov))
        x_fit = np.linspace(x_arr.min(), x_arr.max(), 300)
        y_fit = sigmoid_2p(x_fit, *popt)
        ax.plot(x_fit, y_fit, '-', color=C_FIT, lw=1.6, zorder=5,
                label=f'Sigmoid  $k$={popt[0]:.2f}')
        for dk in (-1.96*perr[0], 1.96*perr[0]):
            y_band = sigmoid_2p(x_fit, popt[0]+dk, popt[1])
            ax.fill_between(x_fit, y_fit, y_band,
                            color=C_FIT, alpha=0.09, linewidth=0)
    except Exception as e:
        print(f"  [A] Sigmoid failed: {e}")

    ax.axhline(1/3, color=C_BOTH, ls=':', lw=0.9, alpha=0.7, label='Chance (0.33)')
    ax.axvline(0,   color=C_BOTH, ls=':', lw=0.6, alpha=0.5)
    ax.set_xlabel(r'Relative composite value ($\Delta V$)', labelpad=3)
    ax.set_ylabel('P(choose)', labelpad=3)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(loc='upper left', handlelength=1.3, borderpad=0.3,
              labelspacing=0.3)
    despine(ax)
    fig.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
#  PANEL B — Conflict trial analysis (equal/unequal comparison)
# ─────────────────────────────────────────────────────────────────────────────

def make_panel_B(trial_df):
    """
    In unequal-condition games where argmax(V) ≠ argmax(Q) (conflict trials):
      - P(chose max-V option) vs P(chose max-Q option)
      - Per-subject paired comparison → Wilcoxon

    This is the correct equal/unequal comparison because:
      · Equal condition:   I is uniform → argmax(V) = argmax(Q) always.
                           No conflict trials exist; information term
                           cancels out and cannot influence choice direction.
      · Unequal condition: I differs → conflict trials emerge.
                           Whether participants follow max-V vs max-Q
                           directly tests information-guided choice.
    """
    fig, ax = plt.subplots(figsize=(2.5, 2.5))

    # Conflict trials only (unequal condition, argmax-V ≠ argmax-Q)
    conflict = trial_df.query("condition == 'unequal' and is_conflict == 1").copy()

    if conflict.empty:
        ax.text(0.5, 0.5,
                'No conflict trials found.\n'
                'Check condition inference\nor I_std threshold.',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='gray')
        despine(ax)
        fig.tight_layout()
        return fig

    # Per-subject P(chose max-V) and P(chose max-Q) in conflict trials
    subj_stats = (conflict.groupby('subject')
                  .agg(p_maxV=('chose_max_V', 'mean'),
                       p_maxQ=('chose_max_Q', 'mean'),
                       n=('chose_max_V', 'count'))
                  .reset_index())
    subj_stats = subj_stats[subj_stats['n'] >= 3]   # require ≥3 conflict trials

    if len(subj_stats) < 3:
        ax.text(0.5, 0.5,
                f'Only {len(subj_stats)} subjects with ≥3 conflict trials.',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=8, color='gray')
        despine(ax)
        fig.tight_layout()
        return fig

    v_vals = subj_stats['p_maxV'].values
    q_vals = subj_stats['p_maxQ'].values

    stat, p = wilcoxon(v_vals, q_vals, alternative='two-sided')
    d       = cohen_d_paired(v_vals, q_vals)

    pos_V, pos_Q = 1.0, 2.0
    raincloud(ax, v_vals, pos_V, C_EQUAL,   width=0.30, jitter=0.04)
    raincloud(ax, q_vals, pos_Q, C_UNEQUAL, width=0.30, jitter=0.04)

    # Paired lines
    rng = np.random.default_rng(0)
    for v, q in zip(v_vals, q_vals):
        jx = rng.uniform(-0.025, 0.025)
        ax.plot([pos_V - 0.20 + jx, pos_Q - 0.20 + jx],
                [v, q],
                color=C_BOTH, lw=0.5, alpha=0.40, zorder=1)

    # Bracket
    y_top = max(v_vals.max(), q_vals.max())
    y_br  = y_top + (y_top - min(v_vals.min(), q_vals.min())) * 0.12
    ax.set_ylim(top=y_br * 1.22)
    bracket(ax, pos_V, pos_Q,
            y_top + (ax.get_ylim()[1] - y_top) * 0.15,
            p, d=abs(d))

    # Group mean bars (thin horizontal lines for readability)
    for pos, vals, color in [(pos_V, v_vals, C_EQUAL),
                              (pos_Q, q_vals, C_UNEQUAL)]:
        ax.plot([pos - 0.25, pos + 0.08], [vals.mean(), vals.mean()],
                color=color, lw=2.0, zorder=6, alpha=0.85)

    ax.set_xticks([pos_V, pos_Q])
    ax.set_xticklabels(['P(chose\nmax-$V$)', 'P(chose\nmax-$Q$)'])
    ax.set_ylabel('Proportion of conflict trials', labelpad=3)
    ax.set_xlim(0.45, 2.70)

    # Annotation explaining what conflict trials are
    ax.text(0.50, 0.04,
            f'Unequal condition,\nconflict trials only\n($N$={len(subj_stats)} subjects)',
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=7.0, color='k', linespacing=1.4,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.0))

    patches = [mpatches.Patch(color=C_EQUAL,   label='max-$V$ option'),
               mpatches.Patch(color=C_UNEQUAL, label='max-$Q$ option')]
    ax.legend(handles=patches, loc='upper right',
              handlelength=0.9, borderpad=0.3)
    despine(ax)
    fig.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
#  PANEL C — Model fit quality
# ─────────────────────────────────────────────────────────────────────────────

def make_panel_C(trial_df):
    fig, ax = plt.subplots(figsize=(2.5, 2.5))

    subj_cp = (trial_df.groupby('subject')['chosen_prob']
               .mean().reset_index()
               .rename(columns={'chosen_prob': 'mean_cp'}))
    cp_vals = subj_cp['mean_cp'].dropna().values

    raincloud(ax, cp_vals, pos=1.0, color=C_PROB, width=0.45, jitter=0.06)

    ax.axhline(1/3, color=C_BOTH, ls='--', lw=1.2, alpha=0.8,
               label='Chance (0.33)')

    _, p  = wilcoxon(cp_vals - 1/3, alternative='greater')
    med   = np.median(cp_vals)
    ax.text(1.38, med,
            f'Median={med:.2f}\n{sig_stars(p)} vs 0.33',
            va='center', ha='left', fontsize=7.5, color=C_PROB,
            linespacing=1.4)

    mean_loglik = np.log(trial_df['chosen_prob'].clip(1e-10)).mean()
    pseudo_r2   = 1 - mean_loglik / np.log(1/3)
    ax.text(0.05, 0.06,
            f'Mean log-lik/trial: {mean_loglik:.2f}\n'
            f'Pseudo-$R^2$: {pseudo_r2:.2f}',
            transform=ax.transAxes, ha='left', va='bottom',
            fontsize=7.0, color='k', linespacing=1.5)

    ax.set_xticks([1.0])
    ax.set_xticklabels(['Participants'])
    ax.set_ylabel('Mean P(chosen | gkRL model)', labelpad=3)
    ax.set_xlim(0.30, 2.30)
    ax.legend(loc='lower right', handlelength=1.3)
    despine(ax)
    fig.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
#  PANEL E — omega_subj distribution
# ─────────────────────────────────────────────────────────────────────────────

def make_panel_E(subj_df):
    fig, ax = plt.subplots(figsize=(2.5, 2.5))

    omega_col = 'omega_subj_mean'
    if omega_col not in subj_df.columns:
        ax.text(0.5, 0.5, 'omega_subj_mean\nnot found',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
        despine(ax); fig.tight_layout(); return fig

    omega = subj_df[omega_col].dropna().values
    if len(omega) < 3:
        ax.text(0.5, 0.5, f'N={len(omega)} — too few subjects',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
        despine(ax); fig.tight_layout(); return fig

    raincloud(ax, omega, pos=1.0, color=C_OMEGA, width=0.48, jitter=0.06)

    _, p = wilcoxon(omega, alternative='greater')
    med  = np.median(omega)
    ax.axhline(0, color=C_BOTH, ls='--', lw=1.0, alpha=0.7,
               label='$\\omega$ = 0')
    ax.text(1.38, med,
            f'Median={med:.2f}\n{sig_stars(p)} vs 0',
            va='center', ha='left', fontsize=7.5, color=C_OMEGA,
            linespacing=1.4)

    ax.set_xticks([1.0])
    ax.set_xticklabels(['Participants'])
    ax.set_ylabel('Information weight ($\\omega$)', labelpad=3)
    ax.set_xlim(0.30, 2.30)
    ax.legend(loc='lower right', handlelength=1.3)
    despine(ax)
    fig.tight_layout()
    return fig


# ─────────────────────────────────────────────────────────────────────────────
#  PANEL F — omega_subj × explore rate
# ─────────────────────────────────────────────────────────────────────────────

def make_panel_F(subj_df):
    fig, ax = plt.subplots(figsize=(2.5, 2.5))

    omega_col = 'omega_subj_mean'
    df = subj_df[[omega_col, 'explore_rate']].dropna()
    if len(df) < 5:
        ax.text(0.5, 0.5, 'Insufficient paired data',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)
        despine(ax); fig.tight_layout(); return fig

    x  = df[omega_col].values
    y  = df['explore_rate'].values
    rs, p = spearmanr(x, y)

    ax.scatter(x, y, color=C_OMEGA, s=28, alpha=0.80,
               linewidths=0, zorder=3)

    slope, intercept = np.polyfit(x, y, 1)
    x_fit = np.linspace(x.min(), x.max(), 200)
    y_fit = slope * x_fit + intercept
    ax.plot(x_fit, y_fit, '-', color=C_OMEGA, lw=1.6, zorder=4)

    rng  = np.random.default_rng(42)
    boot = []
    for _ in range(1000):
        idx = rng.integers(0, len(x), size=len(x))
        s_, i_ = np.polyfit(x[idx], y[idx], 1)
        boot.append(s_ * x_fit + i_)
    boot = np.array(boot)
    ax.fill_between(x_fit,
                    np.percentile(boot, 2.5,  axis=0),
                    np.percentile(boot, 97.5, axis=0),
                    color=C_OMEGA, alpha=0.15, linewidth=0)

    ax.text(0.05, 0.92,
            f'$r_s$ = {rs:.2f},  {sig_stars(p)}',
            transform=ax.transAxes, ha='left', va='top',
            fontsize=8.5, color='k')

    ax.set_xlabel('Information weight ($\\omega$)', labelpad=3)
    ax.set_ylabel('P(chose max-$V$ option)', labelpad=3)
    despine(ax)
    fig.tight_layout()
    return fig

from scipy.stats import wilcoxon, spearmanr
import json

def compute_behavioral_stats(trial_df, subj_df):
    stats = {}

    # ─────────────────────────────────────
    # A. Psychometric (sigmoid slope)
    # ─────────────────────────────────────
    df = trial_df[['delta_V', 'chose_max_V']].dropna()
    x = df['delta_V'].values
    y = df['chose_max_V'].values.astype(float)

    try:
        popt, pcov = curve_fit(sigmoid_2p, x, y,
                               p0=[0.25, 0.0],
                               bounds=([0.001, -50], [5.0, 50]),
                               maxfev=10000)
        stats['psychometric'] = {
            'k': float(popt[0]),
            'k_se': float(np.sqrt(pcov[0, 0])),
            'x0': float(popt[1]),
            'n_trials': int(len(df))
        }
    except:
        stats['psychometric'] = None

    # ─────────────────────────────────────
    # B. Conflict analysis
    # ─────────────────────────────────────
    conflict = trial_df.query("condition=='unequal' and is_conflict==1")

    subj_stats = (conflict.groupby('subject')
                  .agg(p_maxV=('chose_max_V','mean'),
                       p_maxQ=('chose_max_Q','mean'),
                       n=('chose_max_V','count'))
                  .reset_index())

    subj_stats = subj_stats[subj_stats['n'] >= 3]

    v = subj_stats['p_maxV'].values
    q = subj_stats['p_maxQ'].values

    if len(v) > 0:
        stat, p = wilcoxon(v, q)
        d = cohen_d_paired(v, q)

        stats['conflict'] = {
            'n_subjects': int(len(v)),
            'mean_maxV': float(np.mean(v)),
            'mean_maxQ': float(np.mean(q)),
            'wilcoxon_W': float(stat),
            'p_value': float(p),
            'cohen_d': float(d)
        }

    # ─────────────────────────────────────
    # C. Model fit
    # ─────────────────────────────────────
    cp = (trial_df.groupby('subject')['chosen_prob']
          .mean().dropna().values)

    if len(cp) > 0:
        stat, p = wilcoxon(cp - 1/3, alternative='greater')

        stats['model_fit'] = {
            'n_subjects': int(len(cp)),
            'mean_cp': float(np.mean(cp)),
            'median_cp': float(np.median(cp)),
            'wilcoxon_W': float(stat),
            'p_value': float(p)
        }

    # ─────────────────────────────────────
    # D. omega distribution
    # ─────────────────────────────────────
    if 'omega_subj_mean' in subj_df.columns:
        omega = subj_df['omega_subj_mean'].dropna().values

        if len(omega) > 0:
            stat, p = wilcoxon(omega, alternative='greater')

            stats['omega'] = {
                'n_subjects': int(len(omega)),
                'mean': float(np.mean(omega)),
                'median': float(np.median(omega)),
                'wilcoxon_W': float(stat),
                'p_value': float(p)
            }

    # ─────────────────────────────────────
    # E. Correlation
    # ─────────────────────────────────────
    if 'omega_subj_mean' in subj_df.columns:
        df_corr = subj_df[['omega_subj_mean','explore_rate']].dropna()

        if len(df_corr) > 2:
            r, p = spearmanr(df_corr['omega_subj_mean'],
                             df_corr['explore_rate'])

            stats['correlation'] = {
                'n_subjects': int(len(df_corr)),
                'spearman_r': float(r),
                'p_value': float(p)
            }

    return stats
# ─────────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("Individual behavioral panels — PDF + SVG")
    print("=" * 60)

    trial_df, subj_df = load_all_data()
    stats = compute_behavioral_stats(trial_df, subj_df)
    export_stats(stats, FIG_PATH)

    panels = {
        'FigA_psychometric': make_panel_A(trial_df),
        'FigB_conflict':     make_panel_B(trial_df),
        'FigC_model_fit':    make_panel_C(trial_df),
        'FigE_omega_dist':   make_panel_E(subj_df),
        'FigF_omega_corr':   make_panel_F(subj_df),
    }

    for stem, fig in panels.items():
        print(f"\nSaving {stem}…")
        save_panel(fig, stem)
        plt.close(fig)

    print("\n✓ All panels saved to", FIG_PATH)


if __name__ == "__main__":
    main()

Individual behavioral panels — PDF + SVG
Found 25 metric files, 25 param files
Loaded 2,364 trials | 25 subjects
Conflict trials (unequal, argmax-V ≠ argmax-Q): 480

Saving FigA_psychometric…
  → /root/figures/behavioral_final/FigA_psychometric.pdf
  → /root/figures/behavioral_final/FigA_psychometric.svg

Saving FigB_conflict…
  → /root/figures/behavioral_final/FigB_conflict.pdf
  → /root/figures/behavioral_final/FigB_conflict.svg

Saving FigC_model_fit…
  → /root/figures/behavioral_final/FigC_model_fit.pdf
  → /root/figures/behavioral_final/FigC_model_fit.svg

Saving FigE_omega_dist…
  → /root/figures/behavioral_final/FigE_omega_dist.pdf
  → /root/figures/behavioral_final/FigE_omega_dist.svg

Saving FigF_omega_corr…
  → /root/figures/behavioral_final/FigF_omega_corr.pdf
  → /root/figures/behavioral_final/FigF_omega_corr.svg

✓ All panels saved to /root/figures/behavioral_final
